# LAB 08 — Chạy toàn bộ thí nghiệm trên Colab T4

Mở notebook này trong VS Code, chọn kernel **Colab GPU T4**, rồi chạy lần lượt từng cell. Trước đó, dùng Explorer → **Upload to Colab** để đưa `pretrained_classification.py` và `requirements.txt` vào `/content`. Chỉ chạy huấn luyện sau khi cell kiểm tra xác nhận đúng T4.

Full mode dùng IMDb có nhãn theo cấu hình 20.000 train / 5.000 validation / 25.000 test, 3 epochs; chạy các thí nghiệm code 14–19. Nên lưu output và checkpoint vào Google Drive để có thể tiếp tục nếu phiên Colab bị ngắt.

In [8]:
from pathlib import Path
import torch

SCRIPT = Path('/content/pretrained_classification.py')
REQUIREMENTS = Path('/content/requirements.txt')
assert SCRIPT.is_file(), f'Upload pretrained_classification.py to /content: {SCRIPT}'
assert REQUIREMENTS.is_file(), f'Upload requirements.txt to /content: {REQUIREMENTS}'
assert torch.cuda.is_available(), 'Kernel này chưa có GPU CUDA. Hãy chọn Colab GPU T4.'
GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
assert 'T4' in GPU_NAME.upper(), f'Đã nối GPU khác T4: {GPU_NAME}. Chưa chạy thí nghiệm.'
!nvidia-smi

GPU: Tesla T4
Tue Nov 10 16:17:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------

In [9]:
%pip install -r /content/requirements.txt

## Gắn Google Drive

Nếu `drive.mount` không mở được bước đăng nhập trong VS Code, dùng Command Palette → **Colab: Mount Google Drive to Server** rồi chạy lại cell này.

In [10]:
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = Path('/content/drive/MyDrive/NLP/Lab8')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Results/checkpoints will be saved to:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results/checkpoints will be saved to: /content/drive/MyDrive/NLP/Lab8


## Chạy toàn bộ thí nghiệm

Cell dưới chạy cấu hình full và lưu kết quả vào Drive. Lần chạy có thể kéo dài; nếu runtime bị ngắt, kết nối lại cùng T4/Drive rồi chạy lại cell. Các checkpoint hoàn tất sẽ được nhận diện và tái sử dụng theo cấu hình.

In [11]:
import subprocess
import sys

command = [
    sys.executable, '-u', str(SCRIPT),
    '--mode', 'full',
    '--device', 'cuda',
    '--scratch-data-curve',
    '--output-dir', str(OUTPUT_DIR),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)

Running: /usr/bin/python3 -u /content/pretrained_classification.py --mode full --device cuda --scratch-data-curve --output-dir /content/drive/MyDrive/NLP/Lab8


CompletedProcess(args=['/usr/bin/python3', '-u', '/content/pretrained_classification.py', '--mode', 'full', '--device', 'cuda', '--scratch-data-curve', '--output-dir', '/content/drive/MyDrive/NLP/Lab8'], returncode=0)

In [12]:
from pathlib import Path
import pandas as pd

OUT = Path("/content/drive/MyDrive/NLP/Lab8/outputs/full")

status = OUT / "status.csv"
if status.exists():
    display(pd.read_csv(status))

for path in [
    OUT / "initialization_exception.txt",
    *sorted(OUT.glob("exception_*.txt")),
]:
    if path.exists():
        print(f"\n===== {path.name} =====")
        print(path.read_text()[-12000:])

log = OUT / "run.log"
if log.exists():
    print("\n===== Cuối run.log =====")
    print("\n".join(log.read_text(errors="replace").splitlines()[-100:]))

,experiment,status,message
0,14,EXECUTED,"same architecture/tokenizer, validation compar..."
1,15,EXECUTED,encoder requires_grad checked at model constru...
2,16,EXECUTED,all four nested fractions; scratch curve is op...
3,17,EXECUTED,CPU peak memory not measured if no CUDA
4,18,EXECUTED,A=IMDb B=Yelp Polarity; delta=B-A; nB=38000
5,19,EXECUTED,test n=25000; reference=full-pretrained-fine_t...
6,20,GENERATED,"{""FP"": {""available"": 1955, ""requested"": 3, ""mi..."
7,21,GENERATED,thresholds recorded in config; empty groups ha...



===== Cuối run.log =====
2026-11-10 16:01:00,674 INFO HTTP Request: GET https://huggingface.co/api/resolve-cache/models/distilbert/distilbert-base-uncased/12040accade4e8a0f71eabdb258fecc2e7e948be/tokenizer.json?%2Fdistilbert%2Fdistilbert-base-uncased%2Fresolve%2F12040accade4e8a0f71eabdb258fecc2e7e948be%2Ftokenizer.json=&etag=%22949a6f013d67eb8a5b4b5b46026217b888021b88%22 "HTTP/1.1 200 OK"
2026-11-10 16:01:00,918 INFO HTTP Request: HEAD https://huggingface.co/distilbert/distilbert-base-uncased/resolve/12040accade4e8a0f71eabdb258fecc2e7e948be/added_tokens.json "HTTP/1.1 404 Not Found"
2026-11-10 16:01:01,151 INFO HTTP Request: HEAD https://huggingface.co/distilbert/distilbert-base-uncased/resolve/12040accade4e8a0f71eabdb258fecc2e7e948be/special_tokens_map.json "HTTP/1.1 404 Not Found"
2026-11-10 16:01:01,397 INFO HTTP Request: HEAD https://huggingface.co/distilbert/distilbert-base-uncased/resolve/12040accade4e8a0f71eabdb258fecc2e7e948be/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-

In [13]:
import pandas as pd

results_path = OUTPUT_DIR / 'results.csv'
status_path = OUTPUT_DIR / 'outputs' / 'full' / 'status.csv'
results = pd.read_csv(results_path)
display(results[results['run_mode'] == 'full'])
if status_path.is_file():
    display(pd.read_csv(status_path))
print('Results:', results_path)
print('Artifacts:', OUTPUT_DIR / 'outputs' / 'full')

,experiment_id,experiment_name,run_id,run_mode,model_name,initialization,train_mode,seed,train_fraction,train_samples,...,trainable_parameters,run_status,best_epoch,revision,device,gradient_accumulation_steps,config_hash,accuracy_delta_b_minus_a,f1_macro_delta_b_minus_a,encoder_trainable
0,14,Pretrained vs scratch,full-scratch-fine_tuning-f1-l128-9a9bf39f7d19,full,distilbert/distilbert-base-uncased,scratch,fine_tuning,42,1.00,20000,...,66955010,EXECUTED,3,main,cuda,2,9a9bf39f7d19,NaN,NaN,True
1,14,Pretrained vs scratch,full-pretrained-fine_tuning-f1-l128-3af2f76b3f4e,full,distilbert/distilbert-base-uncased,pretrained,fine_tuning,42,1.00,20000,...,66955010,EXECUTED,3,main,cuda,2,3af2f76b3f4e,NaN,NaN,True
2,15,Frozen vs fine-tuning,full-pretrained-frozen-f1-l128-f1e95e760d35,full,distilbert/distilbert-base-uncased,pretrained,frozen,42,1.00,20000,...,592130,EXECUTED,3,main,cuda,2,f1e95e760d35,NaN,NaN,False
3,15,Frozen vs fine-tuning,full-pretrained-fine_tuning-f1-l128-3af2f76b3f4e,full,distilbert/distilbert-base-uncased,pretrained,fine_tuning,42,1.00,20000,...,66955010,EXECUTED,3,main,cuda,2,3af2f76b3f4e,NaN,NaN,True
4,16,Data efficiency,full-pretrained-fine_tuning-f0.1-l128-5947646d...,full,distilbert/distilbert-base-uncased,pretrained,fine_tuning,42,0.10,2000,...,66955010,EXECUTED,3,main,cuda,2,5947646d4296,NaN,NaN,True
5,16,Data efficiency,full-pretrained-fine_tuning-f0.25-l128-bb90d69...,full,distilbert/distilbert-base-uncased,pretrained,fine_tuning,42,0.25,5000,...,66955010,EXECUTED,2,main,cuda,2,bb90d695ae13,NaN,NaN,True
6,16,Data efficiency,full-pretrained-fine_tuning-f0.5-l128-7b9e3339...,full,distilbert/distilbert-base-uncased,pretrained,fine_tuning,42,0.50,10000,...,66955010,EXECUTED,3,main,cuda,2,7b9e3339e44f,NaN,NaN,True
7,16,Data efficiency,full-pretrained-fine_tuning-f1-l128-3af2f76b3f4e,full,distilbert/distilbert-base-uncased,pretrained,fine_tuning,42,1.00,20000,...,66955010,EXECUTED,3,main,cuda,2,3af2f76b3f4e,NaN,NaN,True
8,16,Data efficiency,full-scratch-fine_tuning-f0.1-l128-05bc86b56dd4,full,distilbert/distilbert-base-uncased,scratch,fine_tuning,42,0.10,2000,...,66955010,EXECUTED,3,main,cuda,2,05bc86b56dd4,NaN,NaN,True
9,16,Data efficiency,full-scratch-fine_tuning-f0.25-l128-638f1d990faa,full,distilbert/distilbert-base-uncased,scratch,fine_tuning,42,0.25,5000,...,66955010,EXECUTED,3,main,cuda,2,638f1d990faa,NaN,NaN,True


,experiment,status,message
0,14,EXECUTED,"same architecture/tokenizer, validation compar..."
1,15,EXECUTED,encoder requires_grad checked at model constru...
2,16,EXECUTED,all four nested fractions; scratch curve is op...
3,17,EXECUTED,CPU peak memory not measured if no CUDA
4,18,EXECUTED,A=IMDb B=Yelp Polarity; delta=B-A; nB=38000
5,19,EXECUTED,test n=25000; reference=full-pretrained-fine_t...
6,20,GENERATED,"{""FP"": {""available"": 1955, ""requested"": 3, ""mi..."
7,21,GENERATED,thresholds recorded in config; empty groups ha...


Results: /content/drive/MyDrive/NLP/Lab8/results.csv
Artifacts: /content/drive/MyDrive/NLP/Lab8/outputs/full


In [ ]:
import pandas as pd

full_results = pd.read_csv(OUTPUT_DIR / 'results.csv')
columns = [
    'experiment_id', 'experiment_name', 'initialization', 'train_mode',
    'train_fraction', 'train_samples', 'validation_samples', 'test_samples',
    'max_length', 'split', 'domain', 'accuracy', 'precision_macro',
    'recall_macro', 'f1_macro', 'f1_binary', 'training_time_seconds',
    'peak_gpu_memory_mb', 'total_parameters', 'trainable_parameters',
    'best_epoch', 'device', 'run_status',
]
print('=== FULL RESULTS CSV ===')
print(full_results.loc[full_results['run_mode'] == 'full', columns].to_csv(index=False))

for name in ['error_analysis.md', 'outputs/full/error_candidates.csv',
             'outputs/full/confidence_groups.csv', 'outputs/full/environment.json']:
    path = OUTPUT_DIR / name
    print(f'\n=== {name} ===')
    if path.is_file():
        print(path.read_text(errors='replace'))
    else:
        print('FILE NOT FOUND:', path)
